In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Connect to MotherDuck (automatically uses the MOTHERDUCK_TOKEN exported in your Fish shell)
con = duckdb.connect('md:')

Attempting to automatically open the SSO authorization page in your default browser.
Please open this link to login into your account: https://auth.motherduck.com/activate?user_code=HQXV-LKJN


Token successfully retrieved ✅

You can display the token and store it as an environment variable to avoid having to log in again:
  PRAGMA PRINT_MD_TOKEN;


In [3]:
# Join Fact and Dimensions using DuckDB SQL
query = """
    SELECT 
        f.*,
        d.year, d.month, d.day, d.hour,
        l.latitude, l.longitude
    FROM my_db.main.fact_climate f
    LEFT JOIN my_db.main.dim_datetime d 
        ON f.measured_at = d.measured_at
    LEFT JOIN my_db.main.dim_location l 
        ON f.city = l.city
    ORDER BY f.measured_at DESC
"""

In [4]:
weather_data = con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [21]:
weather_data.to_csv('../results/weather_data.csv', index=False)

In [5]:
display(weather_data.head())

,city,measured_at,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,wind_speed_10m,pm10,pm2_5,nitrogen_dioxide,european_aqi,year,month,day,hour,latitude,longitude
0,Istanbul,2026-09-12 23:00:00,21.7,82.0,23.4,0.0,9.7,13.6,8.7,22.8,37.0,2026,9,12,23,41.01384,28.94966
1,Izmir,2026-09-12 23:00:00,25.8,45.0,26.4,0.0,2.2,26.1,16.8,46.8,52.0,2026,9,12,23,38.41273,27.13838
2,Ankara,2026-09-12 23:00:00,24.3,22.0,22.2,0.0,0.9,28.6,22.1,64.7,62.0,2026,9,12,23,39.91987,32.85427
3,London,2026-09-12 23:00:00,17.9,86.0,17.9,0.5,12.0,9.1,5.9,15.1,27.0,2026,9,12,23,51.50853,-0.12574
4,Tokyo,2026-09-12 23:00:00,21.2,91.0,24.1,0.0,5.3,22.8,21.2,50.1,54.0,2026,9,12,23,35.68950,139.69171


## 1. Extreme weather conditions change through the years

In [12]:
# 1. Aggregate daily extremes per city and year (Focusing on full calendar years 2017-2025)
sql_volatility = """
WITH daily_stats AS (
    SELECT 
        city,
        year,
        CAST(measured_at AS DATE) AS obs_date,
        MAX(temperature_2m) AS max_daily_temp,
        MIN(temperature_2m) AS min_daily_temp,
        AVG(temperature_2m) AS avg_daily_temp
    FROM weather_data
    -- Filter out partial edge years (2016 started in Sept, 2026 ends in Sept) for strict comparison
    WHERE year BETWEEN 2017 AND 2025
    GROUP BY city, year, CAST(measured_at AS DATE)
),
annual_summary AS (
    SELECT 
        city,
        year,
        COUNT(CASE WHEN max_daily_temp >= 35.0 THEN 1 END) AS extreme_heat_days,
        COUNT(CASE WHEN min_daily_temp <= 0.0 THEN 1 END) AS freezing_days,
        ROUND(AVG(avg_daily_temp), 2) AS annual_mean_temp
    FROM daily_stats
    GROUP BY city, year
)
SELECT * FROM annual_summary ORDER BY city, year
"""
df_volatility = duckdb.sql(sql_volatility).df()

In [13]:
# 2. Pivot for Heatmap
pivot_heat = df_volatility.pivot(index="city", columns="year", values="extreme_heat_days").fillna(0)

# Sort cities by total extreme heat days descending so hottest appear on top
pivot_heat = pivot_heat.loc[pivot_heat.sum(axis=1).sort_values(ascending=False).index]

fig_heat = px.imshow(
    pivot_heat,
    labels=dict(x="Year", y="City", color="Days ≥ 35°C"),
    x=pivot_heat.columns,
    y=pivot_heat.index,
    color_continuous_scale="Reds",
    text_auto=True,
    aspect="auto",
    title="<b>Extreme Heat Days Frequency (Max Daily Temp ≥ 35°C) | 2017–2025</b>"
)

fig_heat.update_layout(
    template="plotly_white",
    width=1150,
    height=620,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    xaxis=dict(tickmode="linear", dtick=1, showgrid=False),
    yaxis=dict(showgrid=False),
    coloraxis_colorbar=dict(
        title="Days",
        thickness=18,
        len=0.85
    )
)

# Render with crisp vector configuration and 3x retina download scale
config = {
    "toImageButtonOptions": {"format": "png", "filename": "extreme_heat_heatmap", "scale": 3},
    "responsive": True,
    "scrollZoom": True
}

fig_heat.show(config=config)


In [14]:
# Query the Top 7 cities with the highest temperature spread across the decade
top_volatile_cities_query = """
SELECT 
    city,
    ROUND(MAX(annual_mean_temp) - MIN(annual_mean_temp), 2) AS temp_swing,
    ROUND(STDDEV(annual_mean_temp), 2) AS volatility_score
FROM df_volatility
GROUP BY city
ORDER BY temp_swing DESC
LIMIT 7
"""

top_7_df = duckdb.sql(top_volatile_cities_query).df()
top_7_cities = top_7_df["city"].tolist()

print(f"Top 7 Most Fluctuation-Prone Cities: {top_7_cities}")

# Filter annual data for these 7 cities
df_top7 = df_volatility[df_volatility["city"].isin(top_7_cities)]

fig_line = px.line(
    df_top7,
    x="year",
    y="annual_mean_temp",
    color="city",
    markers=True,
    title="<b>10-Year Temperature Trajectory: Top 7 Most Volatile Cities</b>",
    labels={
        "year": "Year",
        "annual_mean_temp": "Annual Mean Temperature (°C)",
        "city": "Metropolis"
    },
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig_line.update_traces(
    line=dict(width=2.5),
    marker=dict(size=8, symbol="circle")
)

fig_line.update_layout(
    template="plotly_white",
    width=1150,
    height=550,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    hovermode="x unified",
    xaxis=dict(
        tickmode="linear", 
        dtick=1,
        showgrid=True,
        gridcolor="#f0f2f5",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1",
        zeroline=False
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title=None
    )
)

fig_line.show(config=config)

Top 7 Most Fluctuation-Prone Cities: ['Moscow', 'Tehran', 'Tokyo', 'Berlin', 'Stockholm', 'Beijing', 'Madrid']


## 2. The Atmospheric Trap (Weather vs Pollution Correlation)

In [16]:
# 1. Aggregate PM2.5 by Temperature and Wind Speed Buckets
sql_stagnation = """
SELECT 
    FLOOR(wind_speed_10m / 2) * 2 AS wind_speed_bucket,
    FLOOR(temperature_2m / 2) * 2 AS temp_bucket,
    AVG(pm2_5) AS avg_pm25,
    COUNT(*) as observation_hours
FROM weather_data
WHERE pm2_5 IS NOT NULL
GROUP BY wind_speed_bucket, temp_bucket
HAVING COUNT(*) > 100 -- Ensure statistical significance
ORDER BY temp_bucket, wind_speed_bucket
"""

df_stag = duckdb.sql(sql_stagnation).df()

In [17]:
# Pivot for the 2D Contour Heatmap
pivot_stag = df_stag.pivot(index="temp_bucket", columns="wind_speed_bucket", values="avg_pm25")

# 2. Render High-Res Light Mode Density Heatmap
fig_trap = px.imshow(
    pivot_stag,
    labels=dict(x="Wind Speed (km/h)", y="Temperature (°C)", color="Avg PM2.5 (μg/m³)"),
    x=pivot_stag.columns,
    y=pivot_stag.index,
    color_continuous_scale="Turbo",
    origin="lower",
    title="<b>The Atmospheric Trap: PM2.5 Accumulation by Wind & Temperature</b>"
)

fig_trap.update_layout(
    template="plotly_white",
    width=1100,
    height=600,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    xaxis_title="Wind Speed (km/h)",
    yaxis_title="Temperature (°C)"
)

# 3x vector scale configuration for sharp zooming/export
config = {"toImageButtonOptions": {"format": "png", "filename": "atmospheric_trap", "scale": 3}, "responsive": True, "scrollZoom": True}
fig_trap.show(config=config)

In [19]:
# 3. Correlation Scatter: Wind Dispersion vs AQI for High-Pollution Cities
sql_scatter = """
SELECT 
    city,
    wind_speed_10m,
    european_aqi
FROM weather_data
WHERE pm2_5 IS NOT NULL 
  AND city IN ('New Delhi', 'Beijing', 'Jakarta')
USING SAMPLE 5000 
"""
df_scatter = duckdb.sql(sql_scatter).df()

fig_scatter = px.scatter(
    df_scatter,
    x="wind_speed_10m",
    y="european_aqi",
    color="city",
    opacity=0.4,
    trendline="ols",
    title="<b>Dispersion Effect: Wind Speed vs European AQI (Sampled)</b>",
    labels={"wind_speed_10m": "Wind Speed (km/h)", "european_aqi": "European AQI"}
)

fig_scatter.update_layout(
    template="plotly_white",
    width=1100,
    height=550,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827")
)

fig_scatter.show(config=config)

## 3. Seasonal Average temperature change over the decade. 

In [22]:
# 1. Aggregate average temperature by year and meteorological season
sql_seasons = """
WITH season_mapped AS (
    SELECT 
        city,
        year,
        month,
        temperature_2m,
        CASE 
            WHEN month IN (12, 1, 2) THEN 'DJF (Dec-Feb)'
            WHEN month IN (3, 4, 5) THEN 'MAM (Mar-May)'
            WHEN month IN (6, 7, 8) THEN 'JJA (Jun-Aug)'
            WHEN month IN (9, 10, 11) THEN 'SON (Sep-Nov)'
        END AS season_group
    FROM weather_data
    -- Filter out partial edge years (2016 and 2026) for complete seasonal calculations
    WHERE year BETWEEN 2017 AND 2025
),
seasonal_avg AS (
    SELECT 
        city,
        year,
        season_group,
        ROUND(AVG(temperature_2m), 2) AS avg_season_temp
    FROM season_mapped
    GROUP BY city, year, season_group
)
SELECT * FROM seasonal_avg ORDER BY city, year, season_group
"""

df_seasons = duckdb.sql(sql_seasons).df()

In [24]:
# 2. Render 2x2 Faceted Line Chart in High-Res Light Mode
fig_seasons = px.line(
    df_seasons,
    x="year",
    y="avg_season_temp",
    color="city",
    facet_col="season_group",
    facet_col_wrap=2,
    markers=True,
    title="<b>Decadal Seasonal Shift: 10-Year Temperature Trajectories by Meteorological Quarter</b>",
    labels={
        "year": "Year",
        "avg_season_temp": "Average Temperature (°C)",
        "city": "Metropolis",
        "season_group": "Season"
    },
    color_discrete_sequence=px.colors.qualitative.Alphabet
)

# 3. Refine Layout Typography and Architecture
fig_seasons.update_layout(
    template="plotly_white",
    width=1200,
    height=850,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    hovermode="x unified",
    legend=dict(
        title="<b>City</b>",
        orientation="v",
        y=1,
        x=1.02,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# 4. Clean Axes, Gridlines, and Enable Independent Zooming per Season
fig_seasons.update_xaxes(
    tickmode="linear", 
    dtick=1, 
    showgrid=True, 
    gridcolor="#f0f2f5", 
    linecolor="#cbd5e1"
)
fig_seasons.update_yaxes(
    showgrid=True, 
    gridcolor="#e2e8f0", 
    linecolor="#cbd5e1", 
    matches=None,          # Unlinks the axes
    showticklabels=True    # FORCES numbers to appear on the right-side charts
)


# 5. Render with Crisp Vector Configuration and 3x Retina Download Scale
config = {
    "toImageButtonOptions": {"format": "png", "filename": "seasonal_temperature_shift", "scale": 3},
    "responsive": True,
    "scrollZoom": True
}

fig_seasons.show(config=config)


## 4. Thermal Stress Effect - Actual Temperature and Felt Temperature

In [25]:
# 1. Map the thermodynamic interaction between Humidity, Wind Speed, and Perceived Delta
sql_surface = """
SELECT 
    FLOOR(relative_humidity_2m / 5) * 5 AS humidity_bucket,
    FLOOR(wind_speed_10m / 3) * 3 AS wind_speed_bucket,
    ROUND(AVG(apparent_temperature - temperature_2m), 2) AS avg_thermal_delta,
    COUNT(*) AS sample_size
FROM weather_data
WHERE year BETWEEN 2017 AND 2025
GROUP BY 1, 2
HAVING COUNT(*) >= 200
ORDER BY humidity_bucket, wind_speed_bucket
"""

df_surface = duckdb.sql(sql_surface).df()
pivot_surface = df_surface.pivot(index="humidity_bucket", columns="wind_speed_bucket", values="avg_thermal_delta")

In [26]:
# 2. Render Diverging Heatmap in Light Mode
fig_surface = px.imshow(
    pivot_surface,
    labels=dict(x="Wind Speed (km/h)", y="Relative Humidity (%)", color="ΔT (°C)"),
    x=pivot_surface.columns,
    y=pivot_surface.index,
    color_continuous_scale="RdBu_r", # Blue = Wind Chill, White = Neutral, Red = Heat Stress
    color_continuous_midpoint=0.0,
    origin="lower",
    aspect="auto",
    title="<b>Thermal Stress Dynamics: Perceived Temperature Delta (Apparent - Actual)</b>"
)

fig_surface.update_layout(
    template="plotly_white",
    width=1150,
    height=600,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    xaxis=dict(title="<b>Wind Speed (km/h)</b>", showgrid=False),
    yaxis=dict(title="<b>Relative Humidity (%)</b>", showgrid=False),
    coloraxis_colorbar=dict(
        title="<b>ΔT (°C)</b>",
        ticksuffix="°",
        thickness=18,
        len=0.85
    )
)

config = {
    "toImageButtonOptions": {"format": "png", "filename": "thermal_stress_surface", "scale": 3},
    "responsive": True,
    "scrollZoom": True
}

fig_surface.show(config=config)

In [27]:
# Query P95 Heat-Stress and P05 Wind-Chill per Metropolis
sql_city_extremes = """
WITH deltas AS (
    SELECT 
        city,
        apparent_temperature - temperature_2m AS delta
    FROM weather_data
    WHERE year BETWEEN 2017 AND 2025
)
SELECT 
    city,
    ROUND(QUANTILE_CONT(delta, 0.95), 2) AS p95_heat_stress,
    ROUND(QUANTILE_CONT(delta, 0.05), 2) AS p05_wind_chill,
    ROUND(MAX(delta), 2) AS max_heat_spike,
    ROUND(MIN(delta), 2) AS max_cold_drop
FROM deltas
GROUP BY city
ORDER BY p95_heat_stress ASC
"""

df_extremes = duckdb.sql(sql_city_extremes).df()

# Horizontal Diverging Bar Chart
fig_bars = go.Figure()

# Wind Chill (P05) - Negative Delta
fig_bars.add_trace(go.Bar(
    y=df_extremes["city"],
    x=df_extremes["p05_wind_chill"],
    name="Wind-Chill Penalty (5th Percentile)",
    orientation="h",
    marker=dict(color="#2563eb", line=dict(width=1, color="#1d4ed8")),
    text=df_extremes["p05_wind_chill"].apply(lambda x: f"{x:+.1f}°C"),
    textposition="outside"
))

# Heat Stress (P95) - Positive Delta
fig_bars.add_trace(go.Bar(
    y=df_extremes["city"],
    x=df_extremes["p95_heat_stress"],
    name="Heat-Stress Penalty (95th Percentile)",
    orientation="h",
    marker=dict(color="#dc2626", line=dict(width=1, color="#b91c1c")),
    text=df_extremes["p95_heat_stress"].apply(lambda x: f"{x:+.1f}°C"),
    textposition="outside"
))

fig_bars.update_layout(
    template="plotly_white",
    width=1150,
    height=750,
    barmode="overlay",
    font=dict(family="Arial, sans-serif", size=12, color="#1f2937"),
    title="<b>Urban Thermal Vulnerability: 95th Percentile Heat-Stress vs 5th Percentile Wind-Chill</b>",
    title_font=dict(size=18, color="#111827"),
    xaxis=dict(
        title="<b>Perceived Temperature Deviation (Apparent - Actual °C)</b>",
        range=[-10, 10],
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor="#475569",
        showgrid=True,
        gridcolor="#f1f5f9"
    ),
    yaxis=dict(title=None, showgrid=False),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="center",
        x=0.5
    )
)

fig_bars.show(config=config)

# 5. City Pollutions Compared to WHO Standards

In [28]:
 # 1. Calculate WHO 24-Hour Safety Breaches per City
sql_who = """
WITH daily_aqi AS (
    SELECT 
        city,
        CAST(measured_at AS DATE) AS obs_date,
        AVG(pm2_5) AS daily_pm25,
        AVG(nitrogen_dioxide) AS daily_no2,
        AVG(pm10) AS daily_pm10
    FROM weather_data
    WHERE pm2_5 IS NOT NULL
    GROUP BY city, CAST(measured_at AS DATE)
),
breach_counts AS (
    SELECT 
        city,
        COUNT(*) AS total_days_recorded,
        COUNT(CASE WHEN daily_pm25 > 15.0 THEN 1 END) AS pm25_breaches,
        COUNT(CASE WHEN daily_no2 > 25.0 THEN 1 END) AS no2_breaches,
        COUNT(CASE WHEN daily_pm10 > 45.0 THEN 1 END) AS pm10_breaches
    FROM daily_aqi
    GROUP BY city
)
SELECT 
    city,
    ROUND((pm25_breaches * 100.0) / total_days_recorded, 1) AS pm25_violation_pct,
    ROUND((no2_breaches * 100.0) / total_days_recorded, 1) AS no2_violation_pct,
    ROUND((pm10_breaches * 100.0) / total_days_recorded, 1) AS pm10_violation_pct
FROM breach_counts
ORDER BY pm25_violation_pct DESC
"""

df_who = duckdb.sql(sql_who).df()

In [30]:
# 2. Render WHO Compliance Bar Chart
df_melted = df_who.melt(
    id_vars="city", 
    value_vars=["pm25_violation_pct", "no2_violation_pct", "pm10_violation_pct"],
    var_name="pollutant", 
    value_name="violation_percentage"
)

# Refined High-Contrast Color Palette
color_map = {
    "PM2.5 (> 15 µg/m³)": "#e11d48",  # Crimson Red 
    "NO2 (> 25 µg/m³)": "#3b82f6",    # Royal Blue 
    "PM10 (> 45 µg/m³)": "#f59e0b"    # Amber Gold 
}

fig_who = px.bar(
    df_melted,
    x="violation_percentage",
    y="city",
    color="pollutant",
    barmode="group",
    orientation="h",
    title="<b>WHO Air Quality Standard Violations (% of Total Days)</b>",
    labels={
        "violation_percentage": "Days Exceeding WHO Limits (%)", 
        "city": "Metropolis", 
        "pollutant": "24-Hour Threshold"
    },
    color_discrete_map=color_map
)

# Move legend to the top and sort cities cleanly
fig_who.update_layout(
    template="plotly_white",
    width=1150,
    height=850,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    yaxis={'categoryorder': 'total ascending'}, 
    xaxis=dict(showgrid=True, gridcolor="#f1f5f9", ticksuffix="%"),
    legend=dict(
        orientation="h",       
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title=None
    )
)

config = {"toImageButtonOptions": {"format": "png", "filename": "who_violations_sharp", "scale": 3}, "responsive": True}
fig_who.show(config=config)

## 6. Urban Rush Hour Commute & NO2 Emmission

In [31]:
# 1. Aggregate average NO2 by hour and city to map diurnal traffic patterns
sql_rush_hour = """
SELECT 
    city,
    hour,
    ROUND(AVG(nitrogen_dioxide), 2) AS avg_no2
FROM weather_data
WHERE pm2_5 IS NOT NULL
  AND city IN ('London', 'Tokyo', 'New Delhi', 'Jakarta', 'Madrid')
GROUP BY city, hour
ORDER BY city, hour
"""

df_rush = duckdb.sql(sql_rush_hour).df()

In [32]:
# 2. Render the Diurnal Cycle Line Chart in High-Res Light Mode
fig_rush = px.line(
    df_rush,
    x="hour",
    y="avg_no2",
    color="city",
    markers=True,
    title="<b>The Urban Rush Hour Commute: Daily NO2 Pollution Cycles</b>",
    labels={
        "hour": "Hour of Day (24H)", 
        "avg_no2": "Average NO2 (µg/m³)",
        "city": "Metropolis"
    }
)

fig_rush.update_layout(
    template="plotly_white",
    width=1150,
    height=600,
    font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
    title_font=dict(size=18, color="#111827"),
    xaxis=dict(
        tickmode="linear", 
        dtick=1, 
        showgrid=True, 
        gridcolor="#f1f5f9",
        range=[0, 23]
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor="#f1f5f9",
        zeroline=True,
        zerolinecolor="#cbd5e1"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title=None
    )
)

config = {
    "toImageButtonOptions": {"format": "png", "filename": "no2_rush_hour", "scale": 3}, 
    "responsive": True, 
    "scrollZoom": True
}

fig_rush.show(config=config)